## Result filter module - _Attention-Retrieval (AR)_ - Reinforcement Learning / Fine-tuning

## Initialization

In [ ]:
import sys
import os
if 'google.colab' in sys.modules:
	# !pip install --upgrade datasets sentence_transformers
	!pip install sentence_transformers
	from IPython.display import clear_output
	clear_output()
else:
	# if not in 'notebooks' directory, change to it
	if not os.getcwd().endswith('Result-filter-RL'):
		os.chdir('notebooks')
		os.chdir('Result-filter-RL')

In [ ]:
# import numpy as np
# import random
# import torch

# # set random seeds for reproducibility
# seed = 42
# random.seed(seed)
# np.random.seed(seed)
# torch.manual_seed(seed)
# if torch.cuda.is_available():
# 	torch.cuda.manual_seed_all(seed)
# 	# for fully deterministic (seeded) CuDNN behavior (slower), you can also do:
# 	# torch.backends.cudnn.deterministic = True
# 	# torch.backends.cudnn.benchmark = False

## Load the model and dataset

In [ ]:
import torch
# from sentence_transformers import CrossEncoder

# Model selected from the initial evaluation
model_name = 'cross-encoder/ms-marco-MiniLM-L6-v2'  # 22.7M params
# 3.3M downloads on Huggingface last month
model_name_short = 'MiniLM-L6-v2'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = CrossEncoder(model_name, device=device)

# def predict(passages: list[str], query: str) -> list[float]:
# 	# Query the LLM with a list of passages and a query.
# 	# Returns a list of scores for each passage.
# 	scores = model.predict([(query, passage) for passage in passages], batch_size=1)
# 	scores = scores.tolist()
# 	return scores

# from datasets import load_from_disk
# dataset = load_from_disk('coding_dataset')
# train_dataset = dataset['train']
# validation_dataset = dataset['validation']

## Cross-encoder Training

### Prepare the data

In [ ]:
# Preprocess the training dataset
# Create triplets of (query, passage, score) for training
import os
import csv

train_data_file = 'msmarco_coding_train_data.csv'
validation_data_file = 'msmarco_coding_validation_data.csv'

if os.path.exists(train_data_file):
	with open(train_data_file, 'r') as f:
		train_data = [row for row in csv.reader(f)][1:]
else:
	raise NotImplementedError('Please upload the dataset file', train_data_file)
	# train_data = []
	# for val_index, row in enumerate(train_dataset):
	# 	query = row['query']
	# 	passages = row['passages']['passage_text']
	# 	scores = row['passages']['is_selected']
	# 	for passage, score in zip(passages, scores):
	# 		train_data.append([query, passage, score])
	# with open(train_data_file, 'w', newline='') as f:
	# 	writer = csv.writer(f)
	# 	writer.writerow(['query', 'passage', 'score'])  # Write header
	# 	writer.writerows(train_data)

if os.path.exists(validation_data_file):
	with open(validation_data_file, 'r') as f:
		validation_data = [row for row in csv.reader(f)][1:]
else:
	raise NotImplementedError('Please upload the dataset file', validation_data_file)
	# validation_data = []
	# for val_index, row in enumerate(validation_dataset):
	# 	query = row['query']
	# 	passages = row['passages']['passage_text']
	# 	scores = row['passages']['is_selected']
	# 	for passage, score in zip(passages, scores):
	# 		validation_data.append([query, passage, score])
	# with open(validation_data_file, 'w', newline='') as f:
	# 	writer = csv.writer(f)
	# 	writer.writerow(['query', 'passage', 'score'])  # Write header
	# 	writer.writerows(validation_data)

print('Sample:')
for query, passage, score in train_data[:1]:
	print(f'  Query: {query}  \n  Passage: {passage[:50]}...  \n  Score: {score}\n')
print(f'Train data size: {len(train_data)}')
print(f'Validation data size: {len(validation_data)}')

Sample:
  Query: what is an of clause sql  
  Passage: SQL clauses site was designed to help programmers ...  
  Score: 0

Train data size: 25078
Validation data size: 3093


### Cross-encoder fine-tuning

In [ ]:
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

model = CrossEncoder(model_name, num_labels=1, device=device)
# num_labels=1 for regression style

train_samples = [InputExample(texts=[query, passage], label=float(score))
				 for query, passage, score in train_data]
val_samples = [InputExample(texts=[query, passage], label=float(score))
			   for query, passage, score in validation_data]
train_dataloader = DataLoader(train_samples, batch_size=16, shuffle=True)

# loss_fct = torch.nn.BCEWithLogitsLoss()  # works for binary labels
num_epochs = 3
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)  # 10% of total steps
print('Warmup steps:', warmup_steps)

os.environ['WANDB_DISABLED'] = 'true'  # disable Weights & Biases logging
model_save_path = f'msmarco-coding-{model_name_short}'

try:
	loaded_model = CrossEncoder(model_save_path, num_labels=1, device=device)
	if loaded_model:
		model = loaded_model
	print(f'Model loaded. Skipping fine-tuning.')
except:
	print('Fine-tuning the model...')
	model.fit(
		train_dataloader=train_dataloader,
		evaluator=CEBinaryClassificationEvaluator.from_input_examples(
					val_samples, name='val'),
		epochs=num_epochs,
		warmup_steps=warmup_steps,
		optimizer_params={ 'lr': 3e-5 },  # learning rate
		use_amp=True,  # for mixed precision training such as fp16
		output_path=model_save_path,
	)
	# model.save_pretrained(model_save_path)
	model.save(model_save_path)
	print(f'Model saved to {model_save_path}')

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Fine-tuning the model...


Step,Training Loss
500,0.360100
1000,0.140800
1500,0.142800
2000,0.130300
2500,0.121100
3000,0.115000
3500,0.103200
4000,0.104700
4500,0.095100


Model saved to msmarco-coding-cross-encoder-model


### Cross-encoder - RL - **failed**

In [ ]:
from sentence_transformers import CrossEncoder
from trl import AutoModelForSeq2SeqLMWithValueHead, PPOTrainer, PPOConfig
import torch

# Wrap the cross-encoder model in a value-head wrapper
# ppo_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(model_save_path)
ppo_model = CrossEncoder(model_save_path)
tokenizer = ppo_model.tokenizer
if not tokenizer:
	print('No tokenizer found, using default tokenizer for the model.')
	tokenizer = ppo_model.get_tokenizer()
if not tokenizer:
	print('No tokenizer found, loading from Huggingface.')
	from transformers import AutoTokenizer
	tokenizer = AutoTokenizer.from_pretrained(model_name)

ppo_config = PPOConfig(
	learning_rate=1e-6, 
	# adap_kl_ctrl=True, target_kl=6.0
)
ppo_trainer = PPOTrainer(ppo_model, ref_model=None, batch_size=4, args=ppo_config)

# def ndcg_reward(scores, labels, k=10):
# 	order = torch.argsort(scores, dim=1, descending=True)
# 	gains = (2**labels - 1)[torch.arange(scores.size(0)).unsqueeze(1), order][:,:k]
# 	discounts = 1. / torch.log2(torch.arange(k, device=device) + 2)
# 	return (gains*discounts).sum(dim=1)

# for batch in train_dataloader:
# 	query, passages, labels = batch
# 	inputs = tokenizer(query, passages, padding=True, truncation=True,
# 						return_tensors='pt').to(device)
# 	outputs = ppo_trainer.model(**inputs)
# 	rewards = ndcg_reward(outputs.logits.squeeze(-1).detach(), labels.to(device))
# 	ppo_trainer.step(inputs, rewards)

# print('PPO training complete.')
# # Save the final model
# ppo_model_save_path = 'msmarco-coding-ppo-model'
# ppo_trainer.save(ppo_model_save_path)
# print(f'PPO model saved to {ppo_model_save_path}')

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


TypeError: PPOTrainer.__init__() got an unexpected keyword argument 'batch_size'

# Wrong code

## Cross-encoder's LambdaLoss method

In [ ]:
from datasets import load_from_disk
train_dataset = load_from_disk('coding_dataset')['train']

max_docs = 10
def it_to_text_transform(row):
	return {
		'query_id': row['query_id'],
		'doc_ids':  [passage['passage_text'] for passage in row['passages'][:max_docs]],
		'labels':   [passage['is_selected']  for passage in row['passages'][:max_docs]],

		'query':    row['query'],
		'query_type': row['query_type'],
		'wellFormedAnswers': row['wellFormedAnswers'] or [row['query']],
		'passages': [passage['passage_text'] for passage in row['passages'][:max_docs]],
		'answers':  [passage['is_selected']  for passage in row['passages'][:max_docs]],
	}

train_dataset.set_transform(it_to_text_transform)

from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.losses import LambdaLoss, NDCGLoss2PPScheme
from sentence_transformers.cross_encoder.trainer import CrossEncoderTrainer
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments

model = CrossEncoder(model_name, num_labels=1)

args = CrossEncoderTrainingArguments(
	output_dir='checkpoints/LambdaLoss',
	# Optional training parameters:
	num_train_epochs=1,
	per_device_train_batch_size=16,
	per_device_eval_batch_size=16,
	learning_rate=2e-5,
	warmup_ratio=0.1,
	fp16=False,
	bf16=True,
	metric_for_best_model='eval_NanoBEIR_R100_mean_ndcg@10',
	# eval_strategy='steps',
	# eval_steps=1000,
	# save_strategy='steps',
	# save_steps=1000,
	# save_total_limit=2,
	load_best_model_at_end=False,
	logging_steps=200,
	logging_first_step=True,
	seed=12,
)

loss = LambdaLoss(model=model, weighting_scheme=NDCGLoss2PPScheme(), 
					mini_batch_size=16)
trainer = CrossEncoderTrainer(
	model=model,
	args=args,
	train_dataset=train_dataset,
	loss=loss,
	# eval_dataset=eval_dataset,
	# evaluator=evaluator,
)
trainer.train()

model_save_path = f'msmarco-coding-{model_name_short}-LambdaLoss'

model.save(model_save_path)
print(f'Model saved to {model_save_path}')

ft_accuracy = evaluate_model(model, model_name_short=None)
if ft_accuracy > base_accuracy:
	print(f'GOOD. Fine-tuned model is better than the base model.')
else:
	print(f'BAD. Fine-tuned model is worse than the base model.')
print(f'                  ({base_accuracy*100:.2f} -> {ft_accuracy*100:.2f})')

## RL using DPO - 1

In [ ]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch
# from torch.nn import functional as F
# from torch.utils.data import Dataset, DataLoader
# import random
# from torch.optim import AdamW

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name)
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model.to(device)

# class PreferenceDataset(Dataset):
# 	def __init__(self, raw_data):
# 		self.samples = []
# 		for row in raw_data:
# 			query = row['query']
# 			passages = row['passages']['passage_text']
# 			labels = row['passages']['is_selected']
# 			positives = [p for p, l in zip(passages, labels) if l == 1]
# 			negatives = [p for p, l in zip(passages, labels) if l == 0]
# 			for pos in positives:
# 				if negatives:
# 					neg = random.choice(negatives)
# 					self.samples.append((query, pos, neg))
	
# 	def __len__(self):
# 		return len(self.samples)

# 	def __getitem__(self, idx):
# 		query, pos, neg = self.samples[idx]
# 		return query, pos, neg

# def dpo_loss(model, tokenizer, query, pos, neg, beta=0.1):
# 	# Concatenate [query, pos] and [query, neg]
# 	pos_inputs = tokenizer(query, pos, padding=True, truncation=True, return_tensors='pt').to(device)
# 	neg_inputs = tokenizer(query, neg, padding=True, truncation=True, return_tensors='pt').to(device)

# 	with torch.no_grad():
# 		pos_logits = model(**pos_inputs).logits
# 		neg_logits = model(**neg_inputs).logits

# 	pos_reward = pos_logits.squeeze()
# 	neg_reward = neg_logits.squeeze()

# 	# DPO loss = -log sigmoid(beta * (pos - neg))
# 	loss = -F.logsigmoid(beta * (pos_reward - neg_reward)).mean()
# 	return loss

# # Training Loop
# def train_dpo(model, tokenizer, dataset, epochs=3, batch_size=8, lr=2e-5):
# 	dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
# 	optimizer = AdamW(model.parameters(), lr=lr)
# 	model.train()
	
# 	for epoch in range(epochs):
# 		total_loss = 0
# 		for batch in dataloader:
# 			optimizer.zero_grad()
# 			batch_loss = 0
# 			for query, pos, neg in zip(*batch):
# 				loss = dpo_loss(model, tokenizer, query, pos, neg)
# 				batch_loss += loss
# 			batch_loss /= len(query)  # average across samples in batch
# 			batch_loss.backward()
# 			optimizer.step()
# 			total_loss += batch_loss.item()
# 		print(f"Epoch {epoch+1}: loss = {total_loss:.4f}")

# dataset = PreferenceDataset(train_dataset)
# print('Training...')
# train_dpo(model, tokenizer, dataset)
# print('Training complete.')

## RL - using DPO - 2

### Load the model and dataset

In [ ]:
from datasets import Dataset

pref_pairs = []
for row in train_dataset:
	query = row['query']
	passages = row['passages']['passage_text']
	labels = row['passages']['is_selected']
	if not query or not passages or not labels:
		print(f'Warning: Empty query or passages for row: {row}')
		continue
	if len(passages) != len(labels):
		print(f'Warning: Mismatched lengths for query "{query}": {len(passages)} passages, {len(labels)} labels.')
		continue

	pos_passages = [passage for passage, label in zip(passages, labels) if label == 1]
	if not pos_passages:
		continue
	neg_passages = [passage for passage, label in zip(passages, labels) if label == 0]
	for pos in pos_passages:
		for neg in neg_passages:
			pref_pairs.append({ 'query': query, 'chosen': pos, 'rejected': neg })

pref_dataset = Dataset.from_list(pref_pairs)


# Load the model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
ref_model = AutoModelForSequenceClassification.from_pretrained(model_name)

### RL with DPO - 1

In [ ]:
import json, random
from typing import List, Dict

import torch
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import CrossEncoder
from tqdm import tqdm

class PrefDataset(Dataset):
	# Simple Dataset for preference-pair JSONL files.
	def __init__(self, data: List[Dict[str, str]]):
		self.data = data
	def __len__(self):
		return len(self.data)
	def __getitem__(self, idx):
		return self.data[idx]

def load_jsonl(path: str):
	with open(path, encoding='utf-8') as f:
		return [json.loads(l) for l in f if l.strip()]

def collate(batch):
	prompts  = [x['prompt']   for x in batch]
	chosen   = [x['chosen']   for x in batch]
	rejected = [x['rejected'] for x in batch]
	return prompts, chosen, rejected

def dpo_loss(chosen_scores, rejected_scores, beta: float = 0.1):
	# Compute the logistic DPO loss for a batch.
	diff = chosen_scores - rejected_scores
	return -torch.log(torch.sigmoid(beta * diff)).mean()

@torch.no_grad()
def eval_pairwise_acc(model: CrossEncoder, dataloader: DataLoader, device: str):
	model.eval()
	correct, total = 0, 0
	for prompts, chosen, rejected in dataloader:
		pairs_ch = [[p, c] for p, c in zip(prompts, chosen)]
		pairs_re = [[p, r] for p, r in zip(prompts, rejected)]
		ch_scores = model(pairs_ch, convert_to_tensor=True, device=device).squeeze()
		re_scores = model(pairs_re, convert_to_tensor=True, device=device).squeeze()
		correct += (ch_scores > re_scores).sum().item()
		total   += ch_scores.size(0)
	return correct / total

# Define values directly
output_dir = "ft_dpo_checkpoints"
epochs = 3
batch_size = 8
lr = 2e-5
beta = 0.1  # DPO temperature
seed = 42

random.seed(seed)
torch.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

train_data = load_jsonl(train_data_file_jsonl)
val_data = load_jsonl(validation_data_file_jsonl)

train_loader = DataLoader(PrefDataset(train_data),
							batch_size=batch_size,
							shuffle=True,
							collate_fn=collate)
val_loader = DataLoader(PrefDataset(val_data),
						batch_size=batch_size,
						shuffle=False,
						collate_fn=collate)

# Initialize model
model = CrossEncoder(model_name, num_labels=1, device=device)
optimizer = torch.optim.AdamW(model.model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
	optimizer, epochs * len(train_loader))

# Training loop
for epoch in range(1, epochs + 1):
	model.model.train()
	pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
	running_loss = 0.0
	for prompts, chosen, rejected in pbar:
		pairs_ch = [[p, c] for p, c in zip(prompts, chosen)]
		pairs_re = [[p, r] for p, r in zip(prompts, rejected)]

		chosen_scores = model.predict(pairs_ch, convert_to_tensor=True).squeeze()
		rejected_scores = model.predict(pairs_re, convert_to_tensor=True).squeeze()

		loss = dpo_loss(chosen_scores, rejected_scores, beta=beta)
		loss.backward()

		optimizer.step()
		scheduler.step()
		optimizer.zero_grad()

		running_loss += loss.item()
		pbar.set_postfix(loss=running_loss / (pbar.n + 1))

	# Validation
	val_acc = eval_pairwise_acc(model, val_loader, device)
	print(f"Validation pairwise accuracy: {val_acc:.4f}")

	# # Save checkpoint each epoch
	# out_dir = Path(output_dir) / f"epoch{epoch}"
	# out_dir.mkdir(parents=True, exist_ok=True)
	# model.save(str(out_dir))
	# print(f"Model saved to {out_dirmodel.}")


### RL with DPO - 2

In [ ]:
def tokenize_example(example):
	prompt = example['query']

	# Tokenize (query + chosen)
	chosen_inputs = tokenizer(
		prompt,
		example['chosen'],
		truncation=True,
		padding='max_length',
		max_length=512
	)

	# Tokenize (query + rejected)
	rejected_inputs = tokenizer(
		prompt,
		example['rejected'],
		truncation=True,
		padding='max_length',
		max_length=512
	)

	return {
		'prompt_input_ids': chosen_inputs['input_ids'],  # prompt is shared
		'chosen_input_ids': chosen_inputs['input_ids'],
		'rejected_input_ids': rejected_inputs['input_ids']
	}

tokenized_pref_dataset = pref_dataset.map(tokenize_example)

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
	learning_rate=1e-5,
	report_to='none',
	# clip range etc...
)

dpo_trainer = DPOTrainer(
	model=model,
	ref_model=ref_model,
	args=dpo_config,
	train_dataset=tokenized_pref_dataset,
	processing_class=tokenizer,
)

print('Training...')
dpo_trainer.train()
print('Training complete')

if False:
	# Save the trained model
	model.save_pretrained('dpo_trained_model')
	# Save the tokenizer
	tokenizer.save_pretrained('dpo_trained_model')
	# Save the dPOTrainer state
	dpo_trainer.save_state('dpo_trained_model/dpo_trainer_state')
	# Save the reference model
	ref_model.save_pretrained('dpo_trained_model/ref_model')
	# Save the value model
	value_model.save_pretrained('dpo_trained_model/value_model')
	# Save the dPO config
	dpo_config.save_pretrained('dpo_trained_model/dpo_config')
	print('Saved to disk')

## RL - using PPO

In [ ]:
# import torch
# from torch.utils.data import Dataset, DataLoader

# # find the max number of passages any example has
# # N_max = max(len(row['passages']['passage_text']) for row in train_dataset)
# N_max = 10  # There are 6-10 passages per query in the dataset.
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# class MSMarcoDataset(Dataset):
# 	def __init__(self, data):
# 		self.data = list(data)

# 	def __len__(self):
# 		return len(self.data)

# 	def __getitem__(self, idx):
# 		item = self.data[idx]
# 		return {
# 			'query': item['query'],
# 			'passages': item['passages']['passage_text'],
# 			'labels': item['passages']['is_selected'],
# 		}

# def data_collator(batch):
# 	# Pads each example in the batch to the max number of passages in that batch `N_max`
# 	queries = []
# 	padded_passages = []
# 	padded_labels = []

# 	for row in batch:
# 		query    = row['query']
# 		passages = row['passages']
# 		labels   = row['labels']

# 		# pad with empty strings / zeros
# 		pad_count = N_max - len(passages)
# 		passages += [''] * pad_count
# 		labels   += [0]  * pad_count

# 		queries.append(query)
# 		padded_passages.append(passages)
# 		padded_labels.append(labels)

# 	labels_tensor = torch.tensor(padded_labels, dtype=torch.long, device=device)
# 	return {
# 		'queries':  queries,
# 		'passages': padded_passages,
# 		'labels':   labels_tensor,
# 	}

# dataset = MSMarcoDataset(train_dataset)
# dataloader = DataLoader(dataset, batch_size=1,
# 						collate_fn=collate_fn, shuffle=False)

In [ ]:
# from trl import PPOTrainer, PPOConfig

# def compute_mrr(labels: torch.Tensor, scores: torch.Tensor, k: int = 10) -> float:
# 	# sort by descending score
# 	sorted_indices = torch.argsort(scores, descending=True)
# 	topk_labels = labels[sorted_indices][:k]

# 	# find first '1' label
# 	hits = (topk_labels == 1).nonzero(as_tuple=False)
# 	if hits.numel() == 0:
# 		return 0.0
# 	# +1 since ranks are 1-based
# 	first_rank = hits[0].item() + 1
# 	return 1.0 / first_rank

# def compute_reward(batch_labels, batch_actions):
# 	# batch_actions: 0/1 selection masks
# 	# batch_labels: ground-truth 0/1 labels
# 	# here: average MRR over the batch
# 	return torch.tensor([compute_mrr(labels, actions) 
# 						 for labels, actions in zip(batch_labels, batch_actions)])


# import torch.nn as nn
# class RewardModel(nn.Module):
# 	def forward(self, x):
# 		return compute_reward(x['labels'], x['actions'])


# def data_collator(batch):
# 	# batch: list of {'query', 'passages', 'labels'}
# 	B = len(batch)
# 	N_max = max(len(x['passages']) for x in batch)

# 	all_q, all_p, all_lbl = [], [], []
# 	for ex in batch:
# 		ps   = ex['passages'] + [''] * (N_max - len(ex['passages']))
# 		lbls = ex['labels']   + [0] * (N_max - len(ex['labels']))
# 		all_q.extend([ex['query']] * N_max)
# 		all_p.extend(ps)
# 		all_lbl.extend(lbls)

# 	tok = tokenizer(all_q, all_p, return_tensors='pt', padding=True, truncation=True)
# 	# tok['input_ids'].shape == (B*N_max, seq_len)

# 	# reshape labels back to B×N_max
# 	labels = torch.tensor(all_lbl, dtype=torch.long).view(B, N_max)

# 	return {
# 		'input_ids':      tok['input_ids'],
# 		'attention_mask': tok['attention_mask'],
# 		'labels':         labels,
# 	}

# ppo_config = PPOConfig(
# 	learning_rate=1e-5,
# 	# batch_size=8,
# 	# num_ppo_epochs=4,
# 	# stop_token_id=tokenizer.eos_token_id,
# 	report_to=None,
# 	# clip range etc...
# )

# ppo_trainer = PPOTrainer(
# 	model=model,
# 	ref_model=ref_model,
# 	args=ppo_config,
# 	# data_collator=data_collator,
# 	train_dataset=pref_dataset,
# 	processing_class=tokenizer,
# 	# reward_model=RewardModel(),
# )

# print('Training...')
# ppo_trainer.train()
# print('Training complete')

# if False:
# 	# Save the trained model
# 	model.save_pretrained('ppo_trained_model')
# 	# Save the tokenizer
# 	tokenizer.save_pretrained('ppo_trained_model')
# 	# Save the PPOTrainer state
# 	ppo_trainer.save_state('ppo_trained_model/ppo_trainer_state')
# 	# Save the reference model
# 	ref_model.save_pretrained('ppo_trained_model/ref_model')
# 	# Save the value model
# 	value_model.save_pretrained('ppo_trained_model/value_model')
# 	# Save the PPO config
# 	ppo_config.save_pretrained('ppo_trained_model/ppo_config')
# 	print('Saved to disk')

## RL with TRL library

In [ ]:
# import trl  # huggingface’s 🤗trl

# def compute_mrr(predictions, ground_truths):
# 	'''
# 	Compute Mean Reciprocal Rank (MRR) for a list of predictions and ground truths.
# 	predictions: List of predicted scores for each passage.
# 	ground_truths: List of ground truth relevance labels (0 or 1).
# 	'''
# 	mrr = 0.0
# 	for i, (pred, gt) in enumerate(zip(predictions, ground_truths)):
# 		if gt == 1:  # only consider relevant passages
# 			rank = i + 1  # rank starts at 1
# 			mrr += 1 / rank
# 	return mrr / len(ground_truths) if len(ground_truths) > 0 else 0.0

# def your_preprocess(row):
# 	'''
# 	Preprocess the dataset example to convert it into a state-action format.
# 	For this example, we assume the dataset has 'query' and 'passages' fields.
# 	'''
# 	query = row['query']
# 	passages = row['passages']  # assuming passages is a list of strings
# 	return {
# 		'state': query,
# 		'action': passages  # actions are the passages to be scored
# 	}

# ppo_trainer = trl.trainer.PPOTrainer(
# 	model=model_name,
# 	learning_rate=3e-5,
# 	per_device_train_batch_size=4,
# 	ppo_epochs=4,
# 	clip_range=0.2,
# 	dataset=train_dataset,       # your train dataset
# 	preprocess_fn=your_preprocess, # convert to state, action format
# 	compute_rewards_fn=compute_mrr # returns reward per example
# )

# ppo_trainer.train()


## RL

### Load the model and the data

In [ ]:
# import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# class MSMarcoPassageRankingDataset(Dataset):
# 	# A PyTorch Dataset for MS MARCO passage ranking.
# 	def __init__(self, split, max_passages=None):
# 		self.data = split
# 		# determine max number of passages if not given
# 		if max_passages is None:
# 			self.max_passages = max(len(r['passages']['passage_text']) for r in self.data)
# 		else:
# 			self.max_passages = max_passages

# 	def __len__(self):
# 		return len(self.data)

# 	def __getitem__(self, idx):
# 		row = self.data[idx]
# 		query = row['query']
# 		passages = row['passages']['passage_text']
# 		labels = row['passages']['is_selected']
# 		# pad if fewer passages
# 		if len(passages) < self.max_passages:
# 			pad_size = self.max_passages - len(passages)
# 			passages = passages + [''] * pad_size
# 			labels = labels + [0] * pad_size
# 		else:
# 			passages = passages[:self.max_passages]
# 			labels = labels[:self.max_passages]
# 		return query, passages, labels


# temperature = 1.0
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # load dataset
# train_data = MSMarcoPassageRankingDataset(train_dataset)
# train_loader = DataLoader(train_data, batch_size=1, shuffle=True)

# # tokenizer & model
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name)
# model.to(device)

### Perform RL

In [ ]:
# def compute_mrr(ranking: list[int], labels: list[int], k: int = 10) -> float:
# 	# Compute MRR@k for a single ranking.
# 	for rank, idx in enumerate(ranking[:k]):
# 		if labels[idx] == 1:
# 			return 1.0 / (rank + 1)
# 	return 0.0

# # running baseline reward
# baseline = 0.0

# optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

# num_epochs = 3  # number of epochs to train
# for epoch in range(num_epochs):
# 	print(f'Epoch {epoch+1}/{num_epochs}')
# 	for step, (query, passages, labels) in enumerate(train_loader):
# 		# single-example RL
# 		query = query[0]
# 		passages = passages[0]
# 		labels = labels[0]
# 		K = len(passages)

# 		# tokenize all query-passage pairs
# 		pair_list = [(query, p) for p in passages]
# 		encoded = tokenizer(pair_list, padding=True, truncation=True, return_tensors='pt')
# 		encoded = {k: v.to(device) for k, v in encoded.items()}

# 		# forward pass
# 		outputs = model(**encoded)
# 		scores = outputs.logits.squeeze(-1)  # shape [K]

# 		# compute sampling probabilities
# 		probs = torch.softmax(scores / temperature, dim=0)

# 		# sample a permutation without replacement & accumulate log-prob
# 		remaining_probs = probs.clone()
# 		remaining_indices = list(range(K))
# 		ranking = []
# 		log_prob_terms = []
# 		for _ in range(K):
# 			dist = torch.distributions.Categorical(remaining_probs)
# 			choice = dist.sample()
# 			idx = remaining_indices[choice]
# 			ranking.append(idx)
# 			log_prob_terms.append(dist.log_prob(choice))
# 			# remove chosen
# 			remaining_probs = torch.cat([remaining_probs[:choice], remaining_probs[choice+1:]])
# 			remaining_indices.pop(choice)

# 		log_prob = torch.stack(log_prob_terms).sum()

# 		# compute reward
# 		reward = compute_mrr(ranking, labels, k=10)

# 		# update baseline
# 		baseline = 0.9 * baseline + 0.1 * reward

# 		# compute policy-gradient loss
# 		loss = - (reward - baseline) * log_prob

# 		optimizer.zero_grad()
# 		loss.backward()
# 		optimizer.step()

# 		if step % 20 == 0:
# 			print(f' Step {step}/{len(train_loader)} | reward={reward:.4f} | loss={loss.item():.4f}')

# # save final model
# model.save_pretrained('./rl_cross_encoder')
# tokenizer.save_pretrained('./rl_cross_encoder')

# print('Training complete. Model saved to ./rl_cross_encoder')